In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
!uv pip install -U "jax[cuda12]"

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import numpy as np
import sys
import wandb
import time
from ml_collections import config_flags
import tensorflow as tf
from PIL import Image
sys.path.append(".")
import numpy as np
from flask import Flask, request, jsonify
from flask_ngrok import run_with_ngrok
import pyngrok 
import ngrok
import base64
from io import BytesIO
from PIL import Image

# Palivla
from palivla.model_components import ModelComponents
from palivla.inference import run_inference, make_sharding

# Jax imports
import jax
import jax.numpy as jnp
import orbax.checkpoint as ocp

In [ ]:
jax.default_backend()

In [ ]:
physical_devices = tf.config.list_physical_devices('GPU')
tf.config.set_visible_devices(physical_devices, "GPU")
print("VISIBLE DEVICES: ", jax.devices())

In [ ]:
# Load some data
from configs.bridge_cast_config import get_config as bridge_cast_config
from palivla.dataset import make_base_dataset

config = bridge_cast_config()
config.batch_size = 10
train_ds = make_base_dataset(**config.dataset_kwargs.to_dict(), train=True)
def make_training_batch(batch):
    return batch

train_it = map(
    make_training_batch,
    train_ds.batch(config.batch_size).iterator(),
)


In [ ]:
sharding_metadata = make_sharding(config)

checkpoint_dir = ""
checkpoint_step = 0

print("\nLoading model...", checkpoint_dir)
model = ModelComponents.load_static(checkpoint_dir, sharding_metadata, weights_only=True)
manager = ocp.CheckpointManager(checkpoint_dir, options=ocp.CheckpointManagerOptions())
model.load_state(checkpoint_step, manager, weights_only=True)
print("\nModel loaded!")

In [ ]:
batch = next(train_it)

In [ ]:
print(batch.keys())
print(batch["task"].keys())
print(batch["observation"].keys())
print(batch["action"].shape)
print(batch["task"]["language_instruction"].shape)
print(batch["observation"]["image_primary"].shape)
print(batch["observation"]["proprio"].shape)

In [ ]:
batch["task"]

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
batch_size = batch["observation"]["image_primary"].shape[0]

for i in range(batch_size):
    plt.title(batch["task"]["language_instruction"][i])
    plt.imshow(batch["observation"]["image_primary"][i].squeeze())
    plt.show()

In [ ]:
eval_data = model.eval_step(batch)

In [ ]:
for i in range(batch_size):
    print(eval_data['eval_data']["pred_actions"][i])
    print(batch["action"][i])
    
    print("Error: ", np.linalg.norm(eval_data['eval_data']["pred_actions"][i] - batch["action"][i]))
